In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.model_selection import train_test_split

SEED = 42
torch.set_num_threads(4)

data_dir = Path("/kaggle/input/competitions/comp-472-project-fall-2026-concordia")
working_dir = Path("/kaggle/working")

image_train = pd.read_csv(data_dir / "image_train.csv", dtype={"id": "string"})
image_test = pd.read_csv(data_dir / "image_test.csv", dtype={"id": "string"})
image_feature_columns = [f"feature_{i}" for i in range(500)]
image_train_features = image_train[image_feature_columns].to_numpy(float)
image_train_labels = image_train["label"].to_numpy(int)

# Keep sentiment rows in a fixed order before splitting and training.
sentiment_train = pd.read_csv(data_dir / "sentiment_train.csv").sort_values("id")
sentiment_test = pd.read_csv(data_dir / "sentiment_test.csv", dtype={"id": "string"})

In [2]:
IMAGE_CLASS_COUNT = 50
IMAGE_VOTE_NEIGHBORS = 100
LOCAL_SCALE_NEIGHBORS = 10
IMAGE_VOTE_POWER = 18


def normalize_rows(features):
    # Unit-length rows let us compare directions with a dot product.
    row_lengths = np.linalg.norm(features, axis=1, keepdims=True)
    # Leave zero rows at zero.
    return np.divide(
        features, row_lengths,
        out=np.zeros_like(features),
        where=row_lengths > 0,
    )


def distance_weighted_vote(train_labels, neighbor_distances, neighbor_indices):
    neighbor_labels = train_labels[neighbor_indices]
    closest_distance = neighbor_distances.min(axis=1, keepdims=True)
    # The closest neighbor gets weight 1. Power 18 makes farther votes fall off fast.
    vote_weights = (closest_distance / neighbor_distances) ** IMAGE_VOTE_POWER
    class_scores = np.zeros((len(neighbor_labels), IMAGE_CLASS_COUNT))
    for class_label in range(IMAGE_CLASS_COUNT):
        class_scores[:, class_label] = (vote_weights * (neighbor_labels == class_label)).sum(axis=1)
    return class_scores.argmax(axis=1)


def prepare_image_features(train_features, query_features):
    def square_root_ratios(features):
        features = np.asarray(features, dtype=np.float32)
        # Ratios to the smallest positive feature remove a common scale from the row.
        smallest_positive = np.where(features == 0, np.inf, features).min(axis=1, keepdims=True)
        return np.sqrt(np.rint(features / smallest_positive))

    train_features = square_root_ratios(train_features)
    query_features = square_root_ratios(query_features)

    # Fit centering and scaling on the training rows and reuse them for the queries.
    train_mean = train_features.mean(axis=0)
    train_features = train_features - train_mean
    query_features = query_features - train_mean
    feature_scale = np.maximum(train_features.std(axis=0), 1e-6) ** 0.2
    train_features = normalize_rows(train_features / feature_scale)
    query_features = normalize_rows(query_features / feature_scale)

    # Give higher-variance directions more weight, then restore unit row lengths.
    centered_train = train_features - train_features.mean(axis=0)
    covariance = centered_train.T @ centered_train / len(train_features)
    eigenvalues, covariance_directions = np.linalg.eigh(covariance)
    direction_weights = (eigenvalues + 0.1 * eigenvalues.mean()) ** 0.2
    return (
        normalize_rows((train_features @ covariance_directions) * direction_weights),
        normalize_rows((query_features @ covariance_directions) * direction_weights),
    )


def predict_image_labels(train_features, train_labels, query_features):
    train_features, query_features = prepare_image_features(train_features, query_features)
    batch_size = 384

    def mean_local_distance(reference_distance_scale):
        mean_distances = np.empty(len(train_features), dtype=train_features.dtype)
        for start in range(0, len(train_features), batch_size):
            batch = train_features[start:start + batch_size]
            distances = np.maximum(1 - batch @ train_features.T, 1e-6) * reference_distance_scale
            batch_rows = np.arange(len(batch))
            distances[batch_rows, start + batch_rows] = np.inf  # Exclude each row from its own neighborhood.
            closest_distances = np.partition(
                distances, kth=LOCAL_SCALE_NEIGHBORS - 1, axis=1
            )[:, :LOCAL_SCALE_NEIGHBORS]
            mean_distances[start:start + len(batch)] = closest_distances.mean(axis=1)
        return mean_distances

    # Start with a larger scale in denser neighborhoods.
    reference_distance_scale = mean_local_distance(1) ** -0.5
    for _ in range(5):
        # Training-pair distance uses both the row scale and its neighbor's scale.
        local_distances = mean_local_distance(reference_distance_scale) * reference_distance_scale
        reference_distance_scale *= (np.median(local_distances) / local_distances) ** 0.25

    predicted_labels = []
    for start in range(0, len(query_features), batch_size):
        query_batch = query_features[start:start + batch_size]
        # The dot product gives cosine similarity. The floor keeps vote ratios finite.
        distances = np.maximum(1 - query_batch @ train_features.T, 1e-6) * reference_distance_scale
        # Partition finds the closest group without sorting every training row.
        neighbor_indices = np.argpartition(
            distances, kth=IMAGE_VOTE_NEIGHBORS - 1, axis=1
        )[:, :IMAGE_VOTE_NEIGHBORS]
        neighbor_distances = np.take_along_axis(distances, neighbor_indices, axis=1)
        predicted_labels.append(distance_weighted_vote(train_labels, neighbor_distances, neighbor_indices))
    return np.concatenate(predicted_labels)

In [3]:
def make_sentiment_model(input_size):
    return nn.Sequential(
        nn.Linear(input_size, 300),
        nn.ReLU(),
        nn.Linear(300, 64),
        nn.ReLU(),
        nn.Dropout(0.2),
        nn.Linear(64, 3),
    )


def predict_sentiment_labels(train_df, query_df):
    feature_columns = train_df.columns.drop(["id", "label"])
    train_features = torch.tensor(train_df[feature_columns].to_numpy("float32"))
    train_labels = torch.tensor(train_df["label"].to_numpy(), dtype=torch.long)
    query_features = torch.tensor(query_df[feature_columns].to_numpy("float32"))

    torch.manual_seed(SEED)
    model = make_sentiment_model(len(feature_columns))
    shuffle_generator = torch.Generator().manual_seed(SEED)
    loss_function = nn.CrossEntropyLoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=0.01)
    learning_rate_schedule = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=6)

    for _ in range(5):
        model.train()
        shuffled_indices = torch.randperm(len(train_labels), generator=shuffle_generator)
        for batch_indices in shuffled_indices.split(64):
            optimizer.zero_grad(set_to_none=True)
            # Cross-entropy uses the raw class scores.
            class_scores = model(train_features[batch_indices])
            loss = loss_function(class_scores, train_labels[batch_indices])
            loss.backward()
            optimizer.step()
        learning_rate_schedule.step()

    # Turn off dropout and skip gradients when predicting labels.
    model.eval()
    with torch.no_grad():
        return model(query_features).argmax(dim=1).numpy()

In [4]:
# Hold out 20 percent of each class for validation before fitting on all labeled rows.
image_fit_indices, image_val_indices = train_test_split(
    np.arange(len(image_train_labels)), test_size=0.20, random_state=SEED,
    stratify=image_train_labels,
)
image_val_predictions = predict_image_labels(
    image_train_features[image_fit_indices], image_train_labels[image_fit_indices],
    image_train_features[image_val_indices],
)
image_accuracy = np.mean(image_val_predictions == image_train_labels[image_val_indices])

sentiment_fit_rows, sentiment_val_rows = train_test_split(
    sentiment_train, test_size=0.20, random_state=SEED, stratify=sentiment_train["label"]
)
sentiment_val_predictions = predict_sentiment_labels(sentiment_fit_rows, sentiment_val_rows)
sentiment_accuracy = np.mean(sentiment_val_predictions == sentiment_val_rows["label"].to_numpy())

print(f"Image k-NN validation: {image_accuracy:.3%}")
print(f"Sentiment FFNN validation: {sentiment_accuracy:.3%}")
print(f"Mean validation accuracy: {(image_accuracy + sentiment_accuracy) / 2:.3%}")

Image k-NN validation: 67.975%
Sentiment FFNN validation: 66.375%
Mean validation accuracy: 67.175%


In [5]:
# Refit both models on all labeled rows for the final test predictions.
image_test_labels = predict_image_labels(
    image_train_features, image_train_labels, image_test[image_feature_columns].to_numpy(float),
)
sentiment_test_labels = predict_sentiment_labels(sentiment_train, sentiment_test)

image_predictions = pd.DataFrame({"id": image_test["id"], "label": image_test_labels})
sentiment_predictions = pd.DataFrame({"id": sentiment_test["id"], "label": sentiment_test_labels})
predictions = pd.concat([image_predictions, sentiment_predictions], ignore_index=True)

# Preserve the sample submission's ID order and match each ID to one prediction.
sample_submission = pd.read_csv(data_dir / "sample_submission.csv", dtype={"id": "string"})
assert len(predictions) == len(sample_submission) == 10_000
assert set(predictions["id"]) == set(sample_submission["id"])
submission = sample_submission[["id"]].merge(predictions, on="id", how="left", validate="one_to_one")
assert submission["id"].equals(sample_submission["id"]) and submission["label"].notna().all()

output_path = working_dir / "submission.csv"
submission.to_csv(output_path, index=False)
print(f"Wrote {len(submission):,} predictions to {output_path}")

Wrote 10,000 predictions to submission.csv
